In [ ]:
import pandas as pd
import numpy as np
from preprocessing import preprocess_data

# Load datasets
amazon = pd.read_csv("Amazon.csv")
google = pd.read_csv("GOOG.csv")
netflix = pd.read_csv("NFLX.csv")

# Apply basic preprocessing
amazon_preprocessed = preprocess_data(amazon, "Amazon")
google_preprocessed = preprocess_data(google, "Google")
netflix_preprocessed = preprocess_data(netflix, "Netflix")

# Function to add technical indicators manually
def add_technical_indicators(df, name):
    print(f"\n--- Adding Technical Indicators to {name} Dataset ---")
    
    # Ensure data is sorted by date (oldest to newest)
    df = df.sort_values(by='Date').reset_index(drop=True)
    
    # Simple Moving Averages
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    
    # Exponential Moving Averages
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD (Moving Average Convergence Divergence)
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_signal']
    
    # Bollinger Bands
    df['BB_middle'] = df['SMA_20']
    df['BB_upper'] = df['BB_middle'] + 2 * df['Close'].rolling(window=20).std()
    df['BB_lower'] = df['BB_middle'] - 2 * df['Close'].rolling(window=20).std()
    
    # Drop rows with NaN values introduced by indicators
    df = df.dropna().reset_index(drop=True)
    
    print(f"Technical indicators added. Shape: {df.shape}")
    print(df.head())
    
    return df

# Add indicators to each dataset
amazon_featured = add_technical_indicators(amazon_preprocessed, "Amazon")
google_featured = add_technical_indicators(google_preprocessed, "Google")
netflix_featured = add_technical_indicators(netflix_preprocessed, "Netflix")

# Save featured datasets for later use
amazon_featured.to_csv("Amazon_featured.csv", index=False)
google_featured.to_csv("GOOG_featured.csv", index=False)
netflix_featured.to_csv("NFLX_featured.csv", index=False)

print("\nFeatured datasets saved as CSV files.")